# 📊 Data Tables

> Zero-boilerplate data table with pagination, search, sorting, and inline actions

In [ ]:
#| default_exp datatable

In [ ]:
#| export

from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
from fasthtml.common import A, Button as FhButton, I, Span
from fh_matui.foundations import normalize_tokens, stringify, VEnum, dedupe_preserve_order
from fh_matui.core import *
from nbdev.showdoc import show_doc
from fh_matui.components import *

## 🎯 Overview

| Category | Components | Purpose |
|----------|------------|---------|
| 📋 Table | `DataTable` | Paginated data table with search, sort, and actions |
| 🔧 Resource | `DataTableResource` | Zero-boilerplate class to wire everything together |

> 💡 **Note**: Form components (`FormField`, `FormModal`, `FormGrid`) are in the Components module. Import via `from fh_matui.components import *`

---

## 🏗️ Architecture

```
┌─────────────────────────────────────────────────────────┐
│                   DataTableResource                      │
├─────────────────────────────────────────────────────────┤
│  Auto-registers routes:                                  │
│  ├─ GET /resource       → DataTable (list view)         │
│  ├─ GET /resource/action→ FormModal (create/edit)       │
│  └─ POST /resource/save → Save handler (insert/update)  │
├─────────────────────────────────────────────────────────┤
│  DataTable                                              │
│  ├─ Pagination controls (page size, prev/next)          │
│  ├─ Search input (debounced, HTMX)                      │
│  ├─ Sortable column headers                             │
│  └─ Row action menus (edit, delete, custom)             │
├─────────────────────────────────────────────────────────┤
│  FormModal (from components)                            │
│  ├─ Auto-generates fields from column config            │
│  └─ HTMX submit to save endpoint                        │
└─────────────────────────────────────────────────────────┘
```

---

## 📚 Quick Reference

```python
# Complete CRUD in 5 lines
resource = DataTableResource(
    app=app,
    name='products',
    data_source=lambda: db.products(),
    columns=[{'key': 'name', 'label': 'Name'}, {'key': 'price', 'label': 'Price'}]
)
```

In [ ]:
#| hide
#| eval: false

from fasthtml.jupyter import *
from IPython.display import HTML, Markdown, Image
import socket
import time
import subprocess

def kill_process_on_port(port):
    """Kill any process using the specified port on Windows"""
    try:
        # Find process using the port
        result = subprocess.run(
            f'netstat -ano | findstr :{port}',
            shell=True, capture_output=True, text=True
        )
        
        if result.stdout:
            # Extract PID from netstat output
            lines = result.stdout.strip().split('\n')
            for line in lines:
                if 'LISTENING' in line:
                    pid = line.strip().split()[-1]
                    subprocess.run(f'taskkill /PID {pid} /F', shell=True, capture_output=True)
                    print(f"✓ Killed process {pid} on port {port}")
                    time.sleep(0.5)
                    return True
        return False
    except Exception as e:
        print(f"⚠ Could not kill process on port {port}: {e}")
        return False

def find_available_port(start_port=3333, max_attempts=10):
    """Find an available port starting from start_port"""
    for port in range(start_port, start_port + max_attempts):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('', port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"Could not find an available port in range {start_port}-{start_port+max_attempts}")

# Stop existing server if running
if 'server' in globals(): 
    try:
        server.stop()
        time.sleep(0.5)
    except:
        pass

# Try to kill any process on preferred port, then find available port
preferred_port = 5555
kill_process_on_port(preferred_port)
port = find_available_port(preferred_port)

app = FastHTML(hdrs=MatTheme.blue.headers(title="fastmaterial", mode="dark"))
rt = app.route

try:
    server = JupyUvi(app, port=port)
    preview = partial(HTMX, app=app, port=port)
    print(f"✓ Server running on port {port}")
except Exception as e:
    print(f"✗ Failed to start server: {e}")
    raise

✓ Server running on port 5555


## 📋 DataTable Wrapper

| Component | Purpose |
|-----------|---------|
| `DataTable` | Paginated table with search, sort, and row actions |
| `table_state_from_request` | Extract pagination/search state from request |
| `_action_menu` | Dropdown menu for row actions |

**Features:** HTMX-powered pagination, debounced search, sortable columns, action menus

---

### How It Works

The wrapper expects **pre-paginated data** from your backend. Your SQL backend provides:
- `data`: Current page rows (via `LIMIT/OFFSET`)
- `total`: Total record count (via `COUNT(*)`)

```python
@rt("/my-table")
def my_table(req):
    search, page, page_size = extract_params(req)
    data = db.query(f"SELECT * FROM items LIMIT {page_size} OFFSET {(page-1)*page_size}")
    total = db.query("SELECT COUNT(*) FROM items")
    
    return DataTable(
        data=data,
        total=total,
        page=page,
        page_size=page_size,
        search=search,
        columns=[{"key": "name", "label": "Name"}],
        crud_ops={"create": True, "update": True, "delete": True},
        base_route="/my-table"
    )
```

---

### DataTable Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `data` | `list[dict]` | Current page rows (pre-paginated by backend) |
| `total` | `int` | Total record count (for pagination math) |
| `page` | `int` | Current page number (1-indexed) |
| `page_size` | `int` | Records per page |
| `search` | `str` | Current search term |
| `columns` | `list[dict]` | Column configs: `[{"key": "name", "label": "Name", "searchable": True}]` |
| `crud_ops` | `dict` | Enabled operations: `{"create": True, "update": True, "delete": True}` |
| `base_route` | `str` | Base URL for HTMX requests (e.g., `/crud-products`) |
| `row_id_field` | `str` | Field name for row ID (default: `"id"`) |
| `title` | `str` | Table card header title |
| `container_id` | `str` | HTML id for HTMX targeting (auto-generated if None) |
| `page_sizes` | `list` | Page size options (default: `[5, 10, 20, 50]`) |
| `search_placeholder` | `str` | Placeholder for search input |
| `create_label` | `str` | Label for create button |
| `empty_message` | `str` | Message when no records found |

In [ ]:
#| export
#| code-fold: true

from math import ceil
from urllib.parse import urlencode
from typing import Callable, Optional, Any

# Default page size options
PAGE_SIZES = [5, 10, 20, 50]

def _safe_int(value, default):
    """Safely convert to positive int or return default."""
    try:
        number = int(value)
        return number if number > 0 else default
    except (TypeError, ValueError):
        return default

def table_state_from_request(req, page_sizes=None):
    """
    Extract pagination state from request query params.
    
    Returns dict with: search, page, page_size
    """
    page_sizes = page_sizes or PAGE_SIZES
    params = getattr(req, "query_params", {})
    getter = params.get if hasattr(params, "get") else (lambda key, default=None: params[key] if key in params else default)
    
    search = (getter("search", "") or "").strip()
    page = _safe_int(getter("page", 1), 1)
    page_size = _safe_int(getter("page_size", 10), 10)
    
    if page_size not in page_sizes:
        page_size = page_sizes[0] if page_sizes else 10
    
    return {"search": search, "page": page, "page_size": page_size}


def _page_size_select(current_size: int, search: str, base_route: str, container_id: str, page_sizes: list):
    """Build page size dropdown selector."""
    menu_id = f"{container_id}-page-size-menu"
    options = []
    for size in page_sizes:
        params = urlencode({"search": search, "page_size": size, "page": 1})
        option_cls = "active" if size == current_size else None
        options.append(
            Li(
                f"Show {size}",
                hx_get=f"{base_route}?{params}",
                hx_target=f"#{container_id}",
                hx_push_url="true",
                cls=option_cls
            )
        )
    return Button(
        Span(f"Show {current_size}", cls="small-text grey-text"),
        Icon("arrow_drop_down", cls="small grey-text"),
        Menu(*options, cls="border", id=menu_id),
        cls="transparent small",
        data_ui=f"#{menu_id}"
    )


def _action_menu(
    row: dict,
    row_id: Any,
    crud_ops: dict,
    base_route: str,
    search: str,
    page: int,
    page_size: int,
    container_id: str,
    feedback_id: str
):
    """Build per-row action menu based on enabled CRUD operations."""
    menu_id = f"crud-actions-{row_id}"
    base_query = {"id": row_id, "search": search, "page": page, "page_size": page_size}
    
    def action_item(label: str, icon: str, action: str, confirm: str = None):
        query = urlencode({**base_query, "action": action})
        attrs = {
            "hx_get": f"{base_route}/action?{query}",
            "hx_target": f"#{feedback_id}",
            "hx_swap": "outerHTML"
        }
        if confirm:
            attrs["hx_confirm"] = confirm
        return Li(
            A(
                Icon(icon, cls="tiny"),
                Span(label, cls="max"),
                cls="row middle-align",
                **attrs
            )
        )
    
    items = []
    # View is always available (read operation)
    items.append(action_item("View", "visibility", "view"))
    
    if crud_ops.get("update", False):
        items.append(action_item("Edit", "edit", "edit"))
    
    if crud_ops.get("delete", False):
        items.append(action_item("Delete", "delete", "delete", confirm="Delete this record?"))
    
    return Div(
        Button(
            Icon("more_vert"),
            cls=(ButtonT.text, "circle"),
            data_ui=f"#{menu_id}",
            title="Row actions"
        ),
        Menu(*items, id=menu_id),
        cls="relative"
    )


def DataTable(
    data: list[dict],
    total: int,
    page: int = 1,
    page_size: int = 10,
    search: str = '',
    columns: list[dict] = None,
    crud_ops: dict = None,
    base_route: str = '',
    row_id_field: str = 'id',
    title: str = 'Records',
    container_id: str = None,
    page_sizes: list = None,
    search_placeholder: str = 'Search...',
    create_label: str = 'New Record',
    empty_message: str = 'No records match the current filters.'
):
    "Generic data table with server-side pagination, search, and row actions."
    # Defaults
    crud_ops = crud_ops or {"create": False, "update": False, "delete": False}
    page_sizes = page_sizes or PAGE_SIZES
    container_id = container_id or f"crud-table-{base_route.replace('/', '-').strip('-')}"
    feedback_id = f"{container_id}-feedback"
    
    # Auto-generate columns from first data row if not provided
    if columns is None and data:
        columns = [{"key": k, "label": k.replace("_", " ").title()} for k in data[0].keys()]
    columns = columns or []
    
    # Calculate pagination metadata
    total_pages = max(1, ceil(total / page_size)) if total > 0 else 1
    page = min(max(1, page), total_pages)
    start_index = (page - 1) * page_size + 1 if total else 0
    end_index = min(start_index + page_size - 1, total) if total else 0
    summary = f"{start_index}-{end_index} of {total} records" if total else "No matching records"
    
    base_query = urlencode({"search": search, "page_size": page_size})
    
    # Build table header keys and labels
    header_keys = [col["key"] for col in columns] + ["actions"]
    header_labels = [col.get("label", col["key"]) for col in columns] + [""]
    
    # Build table rows
    table_rows = []
    for row in data:
        row_id = row.get(row_id_field)
        row_dict = {}
        
        for col in columns:
            key = col["key"]
            value = row.get(key, "")
            renderer = col.get("renderer")
            
            if renderer and callable(renderer):
                row_dict[key] = renderer(value, row)
            else:
                row_dict[key] = value
        
        # Add actions column
        row_dict["actions"] = _action_menu(
            row=row,
            row_id=row_id,
            crud_ops=crud_ops,
            base_route=base_route,
            search=search,
            page=page,
            page_size=page_size,
            container_id=container_id,
            feedback_id=feedback_id
        )
        table_rows.append(row_dict)
    
    # Empty state
    if not table_rows:
        empty_row = {col["key"]: "" for col in columns}
        empty_row[columns[0]["key"]] = Span(empty_message, cls="small-text grey-text")
        empty_row["actions"] = ""
        table_rows.append(empty_row)
    
    # Build header content
    header_content = DivFullySpaced(
        H5(title),
        Span(f"{total} total", cls="small-text grey-text")
    )
    
    # Build search and create button row
    toolbar_items = [
        Field(
            Input(
                placeholder=search_placeholder,
                name="search",
                value=search,
                hx_get=f"{base_route}?page_size={page_size}",
                hx_trigger="keyup changed delay:500ms",
                hx_target=f"#{container_id}",
                hx_push_url="true",
                hx_vals='{"page":1}',
                autocomplete="off",
                onfocus="this.parentElement.classList.add('max')",
                onblur="if(!this.value) this.parentElement.classList.remove('max')"
            ),
            cls="border round",
            style="transition: flex-grow 0.3s ease"
        )
    ]
    
    if crud_ops.get("create", False):
        create_query = urlencode({"action": "create", "search": search, "page": page, "page_size": page_size})
        toolbar_items.append(
            Button(
                Icon("add"),
                Span(create_label),
                cls=ButtonT.primary,
                hx_get=f"{base_route}/action?{create_query}",
                hx_target=f"#{feedback_id}",
                hx_swap="outerHTML"
            )
        )
    
    # Build footer with pagination - count on left, paginator centered below table
    footer_content = Div(
        Div(Span(summary, cls="small-text grey-text")),
        Div(
            _page_size_select(page_size, search, base_route, container_id, page_sizes),
            Pagination(
                page,
                total_pages,
                f"{base_route}?{base_query}",
                hx_target=f"#{container_id}"
            ),
            cls="row center-align middle-align small-space"
        ),
        cls="grid"
    )
    
    # Create header label mapping for custom rendering
    label_map = dict(zip(header_keys, header_labels))
    
    # Build the card
    card = Card(
        DivFullySpaced(*toolbar_items, cls="padding"),
        TableFromDicts(
            header_keys,
            table_rows,
            header_cell_render=lambda k: Th(label_map.get(k, k)),
            cls="border"
        ),
        footer=footer_content,
        header=header_content,
        cls="surface-container border round"
    )
    
    return Div(card, Div(id=feedback_id), id=container_id)

## 🔧 DataTableResource

| Component | Purpose |
|-----------|---------|
| `DataTableResource` | High-level class that auto-registers table + forms + routes |

**Features:** Auto-registers routes, handles pagination/search/save, lifecycle hooks, multi-tenant support

---

### DataTableResource Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `app` | `FastHTML` | App instance to register routes |
| `base_route` | `str` | Base URL path (e.g., `/products`) |
| `columns` | `list[dict]` | Column config (same as DataTable) |
| `get_all` | `Callable` | `() -> list` of all records |
| `get_by_id` | `Callable` | `(id) -> record` or None |
| `create` | `Callable` | `(data_dict) -> record` |
| `update` | `Callable` | `(id, data_dict) -> record` |
| `delete` | `Callable` | `(id) -> bool` |
| `title` | `str` | Display title for table |

### Lifecycle Hooks

| Hook | Signature | Purpose |
|------|-----------|---------|
| `on_before_create` | `(data) -> data` | Modify data before create |
| `on_after_create` | `(record) -> None` | Side effects after create |
| `on_before_update` | `(id, data) -> data` | Modify data before update |
| `on_after_update` | `(record) -> None` | Side effects after update |
| `on_before_delete` | `(id) -> bool` | Return False to cancel |
| `on_after_delete` | `(id) -> None` | Side effects after delete |

### Multi-tenant Support

```python
products = DataTableResource(
    ...,
    user_filter=lambda req: {"tenant_id": req.state.tenant_id}
)
```

In [ ]:
#| export

from typing import Callable, Optional, Any, Union
from dataclasses import asdict, is_dataclass
from datetime import datetime
import uuid

def _to_dict(obj: Any) -> dict:
    """Convert dataclass, ORM object, or dict to plain dict."""
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    if is_dataclass(obj):
        return asdict(obj)
    # ORM-style objects with __dict__
    if hasattr(obj, "__dict__"):
        return {k: v for k, v in obj.__dict__.items() if not k.startswith("_")}
    return dict(obj)


class DataTableResource:
    "High-level resource that auto-registers all routes for a data table."
    
    def __init__(
        self,
        app,
        base_route: str,
        columns: list[dict],
        get_all: Callable[[], list],
        get_by_id: Callable[[Any], Any],
        create: Callable[[dict], Any] = None,
        update: Callable[[Any, dict], Any] = None,
        delete: Callable[[Any], bool] = None,
        title: str = "Records",
        row_id_field: str = "id",
        crud_ops: dict = None,
        page_sizes: list = None,
        search_placeholder: str = "Search...",
        create_label: str = "New Record",
        empty_message: str = "No records found.",
        # Lifecycle hooks
        on_before_create: Callable[[dict], dict] = None,
        on_after_create: Callable[[Any], None] = None,
        on_before_update: Callable[[Any, dict], dict] = None,
        on_after_update: Callable[[Any], None] = None,
        on_before_delete: Callable[[Any], bool] = None,
        on_after_delete: Callable[[Any], None] = None,
        # Multi-tenant
        user_filter: Callable = None,
        # Custom generators
        id_generator: Callable[[], Any] = None,
        timestamp_fields: dict = None
    ):
        self.app = app
        self.base_route = base_route.rstrip("/")
        self.columns = columns
        self.get_all = get_all
        self.get_by_id = get_by_id
        self.create_fn = create
        self.update_fn = update
        self.delete_fn = delete
        self.title = title
        self.row_id_field = row_id_field
        self.page_sizes = page_sizes or PAGE_SIZES
        self.search_placeholder = search_placeholder
        self.create_label = create_label
        self.empty_message = empty_message
        
        # Determine CRUD ops from provided functions
        if crud_ops is None:
            self.crud_ops = {
                "create": create is not None,
                "update": update is not None,
                "delete": delete is not None
            }
        else:
            self.crud_ops = crud_ops
        
        # Lifecycle hooks
        self.on_before_create = on_before_create
        self.on_after_create = on_after_create
        self.on_before_update = on_before_update
        self.on_after_update = on_after_update
        self.on_before_delete = on_before_delete
        self.on_after_delete = on_after_delete
        
        # Multi-tenant
        self.user_filter = user_filter
        
        # Generators
        self.id_generator = id_generator
        self.timestamp_fields = timestamp_fields or {}
        
        # Derived IDs
        self.container_id = f"crud-table-{base_route.replace('/', '-').strip('-')}"
        self.feedback_id = f"{self.container_id}-feedback"
        self.modal_id = f"{self.container_id}-modal"
        
        # Register routes
        self._register_routes()
    
    def _register_routes(self):
        """Register all data table routes with the app."""
        rt = self.app.route
        
        # Main table route
        @rt(self.base_route)
        def _table_handler(req):
            return self._handle_table(req)
        
        # Action route (view/edit/create/delete)
        @rt(f"{self.base_route}/action")
        def _action_handler(req):
            return self._handle_action(req)
        
        # Save route (form submission)
        @rt(f"{self.base_route}/save")
        async def _save_handler(req):
            return await self._handle_save(req)
    
    def _get_filtered_data(self, req) -> list:
        """Get all data, optionally filtered by user_filter."""
        all_data = self.get_all()
        
        # Convert to list of dicts
        data = [_to_dict(item) for item in all_data]
        
        # Apply user filter if provided
        if self.user_filter and callable(self.user_filter):
            filter_criteria = self.user_filter(req)
            if filter_criteria:
                data = [
                    row for row in data
                    if all(row.get(k) == v for k, v in filter_criteria.items())
                ]
        
        return data
    
    def _filter_by_search(self, data: list, search: str) -> list:
        """Filter data by search term across searchable columns."""
        if not search:
            return data
        
        needle = search.lower()
        searchable_keys = [
            col["key"] for col in self.columns 
            if col.get("searchable", False)
        ]
        
        # If no columns marked searchable, search all
        if not searchable_keys:
            searchable_keys = [col["key"] for col in self.columns]
        
        return [
            row for row in data
            if any(needle in str(row.get(key, "")).lower() for key in searchable_keys)
        ]
    
    def _paginate(self, data: list, page: int, page_size: int) -> tuple:
        """Paginate data list. Returns (page_rows, total, adjusted_page)."""
        total = len(data)
        total_pages = max(1, ceil(total / page_size))
        page = min(max(1, page), total_pages)
        
        start = (page - 1) * page_size
        end = start + page_size
        
        return data[start:end], total, page
    
    def _handle_table(self, req):
        """Handle main table route."""
        # Extract pagination state
        state = table_state_from_request(req, page_sizes=self.page_sizes)
        search, page, page_size = state["search"], state["page"], state["page_size"]
        
        # Get and filter data
        data = self._get_filtered_data(req)
        filtered = self._filter_by_search(data, search)
        
        # Paginate
        page_data, total, page = self._paginate(filtered, page, page_size)
        
        # Build table
        table = DataTable(
            data=page_data,
            total=total,
            page=page,
            page_size=page_size,
            search=search,
            columns=self.columns,
            crud_ops=self.crud_ops,
            base_route=self.base_route,
            row_id_field=self.row_id_field,
            title=self.title,
            container_id=self.container_id,
            page_sizes=self.page_sizes,
            search_placeholder=self.search_placeholder,
            create_label=self.create_label,
            empty_message=self.empty_message
        )
        
        return table
    
    def _handle_action(self, req):
        """Handle action route (view/edit/create/delete)."""
        params = getattr(req, "query_params", {})
        getter = params.get if hasattr(params, "get") else (lambda k, d=None: params[k] if k in params else d)
        
        # Handle dismiss
        if getter("dismiss") is not None:
            return Div(id=self.feedback_id)
        
        record_id = getter("id")
        # Try to convert to int if numeric
        if record_id:
            try:
                record_id = int(record_id)
            except (TypeError, ValueError):
                pass  # Keep as string (e.g., UUID)
        
        action = (getter("action", "view") or "view").lower()
        search = getter("search", "") or ""
        page = _safe_int(getter("page", 1), 1)
        page_size = _safe_int(getter("page_size", 10), 10)
        
        # URLs
        return_params = urlencode({"search": search, "page": page, "page_size": page_size})
        cancel_url = f"{self.base_route}/action?dismiss=1"
        save_url = f"{self.base_route}/save?{return_params}"
        
        # Handle CREATE
        if action == "create":
            if not self.crud_ops.get("create"):
                return self._error_toast("Create operation not enabled.")
            
            modal = FormModal(
                columns=self.columns,
                mode="create",
                record=None,
                modal_id=self.modal_id,
                title=f"New {self.title.rstrip('s')}",
                save_url=save_url,
                save_target=f"#{self.feedback_id}",
                cancel_url=cancel_url,
                cancel_target=f"#{self.feedback_id}"
            )
            return self._wrap_modal(modal)
        
        # Get record for view/edit/delete
        record = None
        if record_id:
            raw_record = self.get_by_id(record_id)
            record = _to_dict(raw_record) if raw_record else None
        
        if not record:
            return self._error_toast("Record not found.")
        
        # Handle VIEW
        if action == "view":
            modal = FormModal(
                columns=self.columns,
                mode="view",
                record=record,
                modal_id=self.modal_id,
                title=f"View {self.title.rstrip('s')}",
                cancel_url=cancel_url,
                cancel_target=f"#{self.feedback_id}"
            )
            return self._wrap_modal(modal)
        
        # Handle EDIT
        if action == "edit":
            if not self.crud_ops.get("update"):
                return self._error_toast("Update operation not enabled.")
            
            modal = FormModal(
                columns=self.columns,
                mode="edit",
                record=record,
                modal_id=self.modal_id,
                title=f"Edit {self.title.rstrip('s')}",
                save_url=save_url,
                save_target=f"#{self.feedback_id}",
                cancel_url=cancel_url,
                cancel_target=f"#{self.feedback_id}"
            )
            return self._wrap_modal(modal)
        
        # Handle DELETE
        if action == "delete":
            if not self.crud_ops.get("delete"):
                return self._error_toast("Delete operation not enabled.")
            
            # Check before_delete hook
            if self.on_before_delete:
                if not self.on_before_delete(record_id):
                    return self._error_toast("Delete cancelled by validation.")
            
            # Perform delete
            try:
                self.delete_fn(record_id)
                
                # After delete hook
                if self.on_after_delete:
                    self.on_after_delete(record_id)
                
                return self._success_toast(f"Record deleted successfully.")
            except Exception as e:
                return self._error_toast(f"Delete failed: {str(e)}")
        
        return Div(id=self.feedback_id)
    
    async def _handle_save(self, req):
        """Handle save route (create/update form submission)."""
        try:
            form_data = await req.form()
            record_id = form_data.get(self.row_id_field)
            
            # Try to convert ID
            if record_id:
                try:
                    record_id = int(record_id)
                except (TypeError, ValueError):
                    pass
            
            # Build data dict from form, excluding the ID field
            data = {}
            for col in self.columns:
                key = col["key"]
                if key == self.row_id_field:
                    continue
                
                form_cfg = col.get("form", {})
                if form_cfg.get("hidden"):
                    continue
                
                value = form_data.get(key)
                
                # Type conversion based on form config
                field_type = form_cfg.get("type", "text")
                if field_type == "number" and value:
                    try:
                        value = float(value) if "." in str(value) else int(value)
                    except (TypeError, ValueError):
                        pass
                elif field_type == "checkbox":
                    value = value == "on" or value == "true" or value == True
                
                data[key] = value
            
            # Add timestamps
            now = datetime.now()
            for field, ts_type in self.timestamp_fields.items():
                if ts_type == "updated":
                    data[field] = now
                elif ts_type == "created" and not record_id:
                    data[field] = now
            
            # CREATE or UPDATE
            if record_id:
                # UPDATE
                if self.on_before_update:
                    data = self.on_before_update(record_id, data)
                
                result = self.update_fn(record_id, data)
                
                if self.on_after_update:
                    self.on_after_update(result)
                
                return self._success_toast("Record updated successfully.")
            else:
                # CREATE
                if self.on_before_create:
                    data = self.on_before_create(data)
                
                # Generate ID if needed
                if self.id_generator:
                    data[self.row_id_field] = self.id_generator()
                
                result = self.create_fn(data)
                
                if self.on_after_create:
                    self.on_after_create(result)
                
                return self._success_toast("Record created successfully.")
        
        except Exception as e:
            return self._error_toast(f"Save failed: {str(e)}")
    
    def _wrap_modal(self, modal):
        """Wrap modal content in feedback div."""
        if isinstance(modal, list):
            return Div(*modal, id=self.feedback_id)
        return Div(modal, id=self.feedback_id)
    
    def _success_toast(self, message: str):
        """Return a success toast in the feedback div."""
        toast = Toast(
            message,
            variant="success",
            position="top",
            action=A(
                "Dismiss",
                cls="inverse-link",
                hx_get=f"{self.base_route}/action?dismiss=1",
                hx_target=f"#{self.feedback_id}",
                hx_swap="outerHTML"
            ),
            active=True
        )
        return Div(toast, id=self.feedback_id)
    
    def _error_toast(self, message: str):
        """Return an error toast in the feedback div."""
        toast = Toast(
            message,
            variant="error",
            position="top",
            action=A(
                "Dismiss",
                cls="inverse-link",
                hx_get=f"{self.base_route}/action?dismiss=1",
                hx_target=f"#{self.feedback_id}",
                hx_swap="outerHTML"
            ),
            active=True
        )
        return Div(toast, id=self.feedback_id)

### DataTableResource Demo

Demonstrates how `DataTableResource` reduces ~100 lines of route handlers to ~15 lines:

In [ ]:
#| code-fold: true
#| eval: false

# --- Mock Data for Demo ---
MOCK_PRODUCTS = [
    {"id": 1, "name": "MacBook Pro 16\"", "category": "Laptops", "price": 2499.00, "stock": 45, "status": "In Stock"},
    {"id": 2, "name": "iPhone 15 Pro", "category": "Phones", "price": 999.00, "stock": 120, "status": "In Stock"},
    {"id": 3, "name": "iPad Air", "category": "Tablets", "price": 599.00, "stock": 0, "status": "Out of Stock"},
    {"id": 4, "name": "AirPods Pro", "category": "Audio", "price": 249.00, "stock": 200, "status": "In Stock"},
    {"id": 5, "name": "Apple Watch Ultra", "category": "Wearables", "price": 799.00, "stock": 30, "status": "Low Stock"},
    {"id": 6, "name": "Magic Keyboard", "category": "Accessories", "price": 299.00, "stock": 85, "status": "In Stock"},
    {"id": 7, "name": "Studio Display", "category": "Monitors", "price": 1599.00, "stock": 12, "status": "Low Stock"},
    {"id": 8, "name": "Mac Mini M2", "category": "Desktops", "price": 599.00, "stock": 60, "status": "In Stock"},
    {"id": 9, "name": "HomePod Mini", "category": "Audio", "price": 99.00, "stock": 150, "status": "In Stock"},
    {"id": 10, "name": "AirTag 4-Pack", "category": "Accessories", "price": 99.00, "stock": 0, "status": "Out of Stock"},
]

# Status chip styles
PRODUCT_STATUS_CLASSES = {
    "In Stock": "chip small success",
    "Low Stock": "chip small warning", 
    "Out of Stock": "chip small error"
}

# Column configuration with renderers and form config
product_columns = [
    {
        "key": "name",
        "label": "Product",
        "searchable": True,
        "renderer": lambda v, row: Strong(v),
        "form": {"type": "text", "required": True}
    },
    {
        "key": "category",
        "label": "Category",
        "searchable": True,
        "form": {
            "type": "select",
            "options": ["Laptops", "Phones", "Tablets", "Audio", "Wearables", "Accessories", "Monitors", "Desktops"]
        }
    },
    {
        "key": "price",
        "label": "Price",
        "renderer": lambda v, row: Span(f"${v:,.2f}", cls="bold"),
        "form": {"type": "number", "min": 0, "step": 0.01}
    },
    {
        "key": "stock",
        "label": "Stock",
        "renderer": lambda v, row: Span(str(v), cls=f"badge {'error' if v == 0 else 'warning' if v < 20 else 'success'}") if isinstance(v, int) else v,
        "form": {"type": "number", "min": 0, "step": 1}
    },
    {
        "key": "status",
        "label": "Status",
        "searchable": True,
        "renderer": lambda v, row: Span(v, cls=PRODUCT_STATUS_CLASSES.get(v, "chip small")),
        "form": {"type": "select", "options": ["In Stock", "Low Stock", "Out of Stock"]}
    }
]

# --- In-memory store (simulates database) ---
DEMO_PRODUCTS = list(MOCK_PRODUCTS)

def demo_get_all():
    return DEMO_PRODUCTS

def demo_get_by_id(id):
    return next((p for p in DEMO_PRODUCTS if p["id"] == id), None)

def demo_create(data):
    new_id = max(p["id"] for p in DEMO_PRODUCTS) + 1
    record = {"id": new_id, **data}
    DEMO_PRODUCTS.append(record)
    return record

def demo_update(id, data):
    for i, p in enumerate(DEMO_PRODUCTS):
        if p["id"] == id:
            DEMO_PRODUCTS[i] = {"id": id, **data}
            return DEMO_PRODUCTS[i]
    return None

def demo_delete(id):
    global DEMO_PRODUCTS
    DEMO_PRODUCTS = [p for p in DEMO_PRODUCTS if p["id"] != id]
    return True

# --- DataTableResource: One object replaces ~100 lines of route handlers ---
products_resource = DataTableResource(
    app=app,
    base_route="/products",
    columns=product_columns,
    get_all=demo_get_all,
    get_by_id=demo_get_by_id,
    create=demo_create,
    update=demo_update,
    delete=demo_delete,
    title="Products",
    search_placeholder="Search products...",
    create_label="Add Product"
)

# --- Preview helper ---
def ex_products():
    """Preview the DataTableResource table."""
    return products_resource._handle_table(type('req', (), {'query_params': {}})())

preview(ex_products())

In [ ]:
#| hide

import nbdev as nb
nb.nbdev_export()